In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_roads = spark.table("dev_catalog.bronze.raw_roads")

In [0]:
category = {
    "TA" :  "Class A Trunk Road",
    "TM" : "Class A Trunk Motorway",
    "PA" : "Class A Principal Road",
    "PM" : "Class A Principal Motorway",
    "M"  : "Class B Road"
}

df_roads = df_roads.withColumn("road_category" , when(~col("road_category").isin(list(category.keys())), lit(None)).otherwise(col("road_category")))



In [0]:
map_expression = create_map([lit(x) for pair in category.items() for x in pair])

df_final = (
    df_roads
    .withColumn("road_category_name", map_expression[col('road_category')])
    .withColumn("expected_road_type" , when(col("road_category_name").contains("Class A") , lit("Major"))
                                        .when (col("road_category_name").contains("Class B") , lit("Minor"))
                                        .otherwise(lit("Unknown")))
    .withColumn("road_type_consistent", when(col("road_type") == col("expected_road_type"), lit(True))
                                        .otherwise(lit(False)))
    )


In [0]:
df_final.write.mode("Overwrite").saveAsTable("dev_catalog.silver.roads")

In [0]:
%sql
  SELECT COUNT(*) FROM dev_catalog.bronze.raw_roads;

